# 진학어플라이 정시 경쟁률 정보 크롤링하기
---
수능 및 입시 시즌이 되면, 많은 분들이 관심을 가지는 정보로 정시 경쟁률을 꼽아볼 수 있습니다. 그래서 수험생들이 경쟁률 정보를 모아 보여주는 [진학어플라이](https://apply.jinhakapply.com) 등의 웹사이트에서 대학별 경쟁률 정보를 찾아보는데, 이걸 한번에 모아서 다운로드하거나 자동으로 수집해서 다양한 기능을 위해 활용해볼 수 있겠습니다. 파이썬(Python)을 어느 정도 아는 분들은 셀레니움(selenium)을 활용해 크롤링하면 간편하게 해결할 수 있습니다.  
</p></br></br>

## 크롤링 대상 살펴보기
---
이번에 데이터를 수집할 대상은 [진학사 스마트경쟁률](https://apply.jinhakapply.com/SmartRatio/)입니다. 여기서는 대학별 입시 경쟁률을 올해 및 지난 연도별로 모아볼 수 있는데요, 저희가 이번에 이용할 데이터는 4년제, 2024년도, 정시 데이터입니다.  
</p></br></br>

<center><img src="fig/jinhak1.png"></center>  
</p></br></br>

위 이미지를 살펴보면, 내가 어떤 선택지를 고르던 URL이 변경되지 않습니다. 그래서 어떤 분들은 접수구분, 학년도, 모집구분 버튼을 일일이 누르고, 그 다음에는 각 학교 정보를 클릭하는 방식으로 프로그램을 설계하기도 하는데, 굳이 그렇게 하지 않아도 됩니다.  
</p></br></br>

## 상세 페이지의 URL 구조 살펴보기
---
저희는 이번에 각 대학별 지난경쟁률 상세페이지를 이용해서 데이터 수집을 진행할겁니다. 지난경쟁률 페이지를 직접 접근할 경우, 앞서 살펴보았던 페이지를 건너뛸 수 있기 때문에 편리하게 크롤링이 가능하기 때문이지요. 해당 페이지는 아래와 같은 URL 구조를 가지고 있습니다.  
</p></br></br>

`https://apply.jinhakapply.com/SmartRatio/PastRatioUniv?univid=1001&year=2024&category=2`  
</p></br></br>

여기서 URL 매개변수를 살펴보면, **univid, year, category** 정보를 확인할 수 있는데요, 각각 학교 번호, 학년도, 모집구분(수시=1, 정시=2) 정보와 대응합니다. 예를 들어, 강남대학교(1005) 2023년(2023) 수시(1)라고 하면 해당 필드에 1005, 2023, 1을 집어넣는 방식이지요. 그런데, 학교 번호와 학교명이 어떻게 대응되는지 확인을 못해서 이번에는 단순히 1001번부터 1100번 데이터를 전부 모으도록 해봤습니다.  
</p></br></br>

<center><img src="fig/jinhak2.png"></center>  
</p></br></br>

적당한 값을 입력하면, 위와 같은 페이지에 접속할 수 있게 됩니다. 이제는 데이터를 잘 정리해서 저장하도록 설계하면 되는데요, 저는 표 형태의 데이터를 곧바로 판다스 데이터프레임(pandas DataFrame)으로 저장할 수 있도록 하겠습니다.  
</p></br></br>

## 에러 대응하기
---
<center><img src="fig/jinhak3.png"></center>
</p></br></br>

그런데, 학교 번호를 있는대로 집어넣을 경우에는 에러 페이지로 리다이렉트되는 경우가 종종 발생합니다. 이 때, 프로그램을 그대로 실행하면 에러가 발생하기 때문에, 페이지 URL을 체크한 뒤 에러 페이지로 리다이렉트되었다면 다음 학교 번호로 이동하도록 합니다.  
</p></br></br>

## 프로그램 구현하기
---
진학어플라이 크롤링 프로그램의 동작 원리는, 셀레니움으로 해당 페이지 접속 - HTML 소스코드 다운로드 - HTML 파싱 - table 태그 찾기 - 판다스 데이터프레임으로 변환 순서로 실행하는 방식입니다.  
</p></br></br>


In [1]:
from selenium import webdriver
from bs4 import BeautifulSoup
import pandas as pd

driver = webdriver.Chrome()
univ_df = pd.DataFrame()
year = 2024
category = 2

for univ_id in range(1001,1100):
    URL = f'https://apply.jinhakapply.com/SmartRatio/PastRatioUniv?univid={univ_id}&year={year}&category={category}'
    driver.get(URL)

    if driver.current_url == 'https://apply.jinhakapply.com/Error/CommonError':
        continue
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # 모집 인원과 지원 인원 분리
    for span in soup.find_all("span", class_="support"):
        span.insert(0, "/")

    # 표 데이터 찾기
    _table_tag = str(soup.find_all('table')[0])
    html_table = pd.read_html(_table_tag)[0]

    if html_table.empty:
        continue
    
    html_table['대학교 URL'] = univ_url = soup.find_all('a','btn_notice')[0]['href']
    univ_df = pd.concat([univ_df, html_table])

In [2]:
univ_df

,전형명 모집단위,모집 인원 /지원 인원,경쟁률,대학교 URL
0,간호학과 기초생활수급자(정원외),1/1,1.00,http://www.kaya.ac.kr
1,물리치료학과 기초생활수급자(정원외),3/3,1.00,http://www.kaya.ac.kr
2,간호학과 농어촌학생(정원외),1/3,3.00,http://www.kaya.ac.kr
3,간호학과 일반학생(정원내),43/162,3.77,http://www.kaya.ac.kr
4,물리치료학과 일반학생(정원내),7/36,5.14,http://www.kaya.ac.kr
...,...,...,...,...
62,회계학과 일반학생전형(정원내),1/4,4.00,https://www.scnu.ac.kr/iphak/main.do
63,기계우주항공공학부 특성화고교졸업자전형(정원외),1/0,0.00,https://www.scnu.ac.kr/iphak/main.do
64,식품공학과 특성화고교졸업자전형(정원외),1/0,0.00,https://www.scnu.ac.kr/iphak/main.do
65,환경공학과 특성화고교졸업자전형(정원외),1/1,1.00,https://www.scnu.ac.kr/iphak/main.do
